# Cross-Modal Validation Strategy — Path A (Pseudo-Paired Fusion)

**Team93 — Phase 3, Task: Cross-Modal Validation Strategy**

### Why this notebook exists
RGB, Thermal, and Plantar datasets come from three different sources with three different label sets:

| Modality | Labels | Source dataset |
|---|---|---|
| RGB | No DFU (0) / DFU (1) | DFU wound image dataset |
| Thermal | Control Group / DM Group | Diabetic foot thermogram dataset |
| Plantar | flat foot / normal / over-arch | Foot pressure / arch dataset |

There is **no patient-level pairing** across these three sources — they were never collected from the same subjects. Late fusion therefore cannot be "true" multimodal fusion of one patient's three scans. This notebook implements **Path A**: a documented, clinically-informed **pseudo-pairing** strategy that builds synthetic multimodal "cases" for fusion training/evaluation, using RGB's DFU label as the fused ground truth.

**This is a stated design assumption, not measured patient data — say so explicitly in the Phase 3 report.** True paired-data acquisition (contacting dataset authors, per your Phase 2 roadmap) remains the Path B stretch goal.

### What this notebook produces
- `paired_train.csv` and `paired_holdout.csv` — synthetic multimodal case tables linking one RGB image + one Thermal feature row + one Plantar feature row per synthetic case, with `fused_label` = the RGB DFU label.
- These feed directly into the Phase 4 late-fusion notebook: for each row, run RGB image through EfficientNet-B3, Thermal features through the tuned RF, Plantar features through the tuned XGBoost, and combine the three probability outputs.

### Split integrity rule
Pairing is only ever done **within matching splits** (train-with-train, holdout-with-holdout). Never mixes a train-split image from one modality with a holdout-split row from another — that would leak information into fusion-weight optimization.

In [6]:
!unzip /content/RGB_P.zip

Archive:  /content/RGB_P.zip
replace RGB_P/Patches/Abnormal(Ulcer)/1.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [7]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit
from collections import Counter

SEED = 42
rng = np.random.default_rng(SEED)

## 1. Config — clinically-informed conditional pairing probabilities

These are **documented assumptions**, chosen to reflect the general clinical relationship between diabetic status, foot structural abnormality, and DFU risk (diabetes and abnormal foot loading are established DFU risk factors), not measured co-occurrence rates from real paired patients. Report this table verbatim in the Phase 3 write-up as a stated limitation.

Adjust these numbers if your team has literature-backed figures to use instead.

In [8]:
# P(Thermal group | RGB label)
# NOTE: keys must match thermal_features.csv's actual 'label' column values exactly
# (the zip's folder names are 'Control Group'/'DM Group', but the extracted CSV
# stores them as 'Control'/'DM' -- confirmed from RF.ipynb's own load cell).
THERMAL_COND_PROB = {
    1: {"DM": 0.85, "Control": 0.15},   # DFU present  -> mostly DM group
    0: {"DM": 0.30, "Control": 0.70},   # DFU absent   -> mostly Control
}

# P(Plantar arch type | RGB label)
PLANTAR_COND_PROB = {
    1: {"flat foot": 0.40, "over-arch": 0.40, "normal": 0.20},  # DFU present  -> mostly abnormal arch
    0: {"flat foot": 0.15, "over-arch": 0.15, "normal": 0.70},  # DFU absent   -> mostly normal arch
}

for label, dist in THERMAL_COND_PROB.items():
    assert abs(sum(dist.values()) - 1.0) < 1e-9, f"Thermal probs for label {label} must sum to 1"
for label, dist in PLANTAR_COND_PROB.items():
    assert abs(sum(dist.values()) - 1.0) < 1e-9, f"Plantar probs for label {label} must sum to 1"

## 2. Load RGB metadata and reproduce the exact EffNet.ipynb split

This mirrors `collect_images()` and the `StratifiedShuffleSplit(seed=42)` from `EffNet.ipynb` exactly, so pairing uses the **same train/val partition** your EfficientNet-B3 model was trained and validated on.

In [9]:
RGB_BASE_DIR = "/content/RGB_P"

RGB_LABEL_MAP = {
    "Abnormal(Ulcer)": 1,
    "Normal(Healthy skin)": 0,
    "Wound Images": 1,
    "Wound Images2": 1,
    "internetSet": 1,
    "samples": 1,
}


def collect_images(base_dir, label_map):
    image_paths, labels = [], []
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}
    for root, dirs, files in os.walk(base_dir):
        folder_name = os.path.basename(root)
        if folder_name not in label_map:
            continue
        label = label_map[folder_name]
        for fname in files:
            if os.path.splitext(fname)[1].lower() in valid_exts:
                image_paths.append(os.path.join(root, fname))
                labels.append(label)
    return image_paths, labels


rgb_paths, rgb_labels = collect_images(RGB_BASE_DIR, RGB_LABEL_MAP)
print(f"RGB images collected: {len(rgb_paths)}")
print(f"  Label distribution: {Counter(rgb_labels)}")

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
indices = list(range(len(rgb_paths)))
train_idx, val_idx = next(sss.split(indices, rgb_labels))

rgb_df = pd.DataFrame({
    "rgb_path": rgb_paths,
    "rgb_label": rgb_labels,
})
rgb_df["split"] = "train"
rgb_df.loc[val_idx, "split"] = "holdout"

print(f"\nRGB split sizes: {rgb_df['split'].value_counts().to_dict()}")

RGB images collected: 1495
  Label distribution: Counter({1: 1225, 0: 270})

RGB split sizes: {'train': 1196, 'holdout': 299}


## 3. Load Thermal and Plantar feature tables, unify split naming

Thermal uses `train`/`val`, Plantar uses `train`/`test` — both are renamed to `train`/`holdout` here so the three modalities share one consistent split vocabulary.

In [10]:
thermal_df = pd.read_csv("/content/thermal_features.csv")
thermal_df["split"] = thermal_df["split"].replace({"val": "holdout"})

plantar_df = pd.read_csv("/content/plantar_features.csv")
plantar_df["split"] = plantar_df["split"].replace({"test": "holdout"})

print(f"Thermal rows: {len(thermal_df)} | splits: {thermal_df['split'].value_counts().to_dict()}")
print(f"Plantar rows: {len(plantar_df)} | splits: {plantar_df['split'].value_counts().to_dict()}")

THERMAL_FEATURE_COLS = [
    "mean_temp", "max_temp", "std_temp", "range_temp",
    "hot_spot_count", "temp_gradient",
    "glcm_contrast", "glcm_homogeneity", "glcm_entropy",
]
PLANTAR_FEATURE_COLS = [
    "mean_pressure", "max_pressure", "contact_area", "cop_x", "cop_y",
    "forefoot_ratio", "midfoot_ratio", "hindfoot_ratio",
    "glcm_contrast", "glcm_homogeneity", "glcm_entropy",
]

Thermal rows: 8942 | splits: {'train': 8520, 'holdout': 422}
Plantar rows: 4962 | splits: {'train': 4621, 'holdout': 341}


## 4. Pairing generator

For every RGB image in a given split, sample one Thermal row and one Plantar row from the same split whose class matches a draw from the conditional probability tables above. Sampling is **without replacement** until a class pool is exhausted, then falls back to **with replacement** (since modality dataset sizes differ substantially) — a `used_with_replacement` flag is logged per row for transparency.

In [11]:
def build_pool(df, class_col):
    """index pool per class, as lists we can pop from for without-replacement sampling"""
    pool = {cls: sub_df.index.tolist() for cls, sub_df in df.groupby(class_col)}
    for cls in pool:
        rng.shuffle(pool[cls])
    return pool


def refill_pool_for_class(df, class_col, cls, rng):
    idxs = df[df[class_col] == cls].index.tolist()
    rng.shuffle(idxs)
    return idxs


def generate_pseudo_paired_split(rgb_split_df, thermal_split_df, plantar_split_df, split_name, rng):
    thermal_pool = build_pool(thermal_split_df, "label")
    plantar_pool = build_pool(plantar_split_df, "label")

    rows = []
    for _, rgb_row in rgb_split_df.iterrows():
        rgb_label = rgb_row["rgb_label"]

        # --- draw thermal class then instance ---
        thermal_classes = list(THERMAL_COND_PROB[rgb_label].keys())
        thermal_probs = list(THERMAL_COND_PROB[rgb_label].values())
        thermal_cls = rng.choice(thermal_classes, p=thermal_probs)

        if len(thermal_pool.get(thermal_cls, [])) == 0:
            thermal_pool[thermal_cls] = refill_pool_for_class(thermal_split_df, "label", thermal_cls, rng)
            thermal_used_wr = True
        else:
            thermal_used_wr = False
        thermal_idx = thermal_pool[thermal_cls].pop()

        # --- draw plantar class then instance ---
        plantar_classes = list(PLANTAR_COND_PROB[rgb_label].keys())
        plantar_probs = list(PLANTAR_COND_PROB[rgb_label].values())
        plantar_cls = rng.choice(plantar_classes, p=plantar_probs)

        if len(plantar_pool.get(plantar_cls, [])) == 0:
            plantar_pool[plantar_cls] = refill_pool_for_class(plantar_split_df, "label", plantar_cls, rng)
            plantar_used_wr = True
        else:
            plantar_used_wr = False
        plantar_idx = plantar_pool[plantar_cls].pop()

        rows.append({
            "split": split_name,
            "rgb_path": rgb_row["rgb_path"],
            "rgb_label": rgb_label,
            "thermal_row_idx": thermal_idx,
            "thermal_label": thermal_cls,
            "thermal_used_with_replacement": thermal_used_wr,
            "plantar_row_idx": plantar_idx,
            "plantar_label": plantar_cls,
            "plantar_used_with_replacement": plantar_used_wr,
            "fused_label": rgb_label,   # ground truth for the fused system = RGB's DFU label
        })

    return pd.DataFrame(rows)

In [12]:
# Guard: fail loudly (not with a bare IndexError) if the config keys above don't
# match the actual label strings present in the loaded CSVs.

def check_labels_match(name, df, cond_prob_config):
    actual_labels = set(df["label"].unique())
    configured_labels = set()
    for dist in cond_prob_config.values():
        configured_labels.update(dist.keys())
    missing = configured_labels - actual_labels
    if missing:
        raise ValueError(
            f"{name}: conditional-probability config references label(s) {missing} "
            f"not present in the data. Actual labels found: {sorted(actual_labels)}. "
            f"Fix the THERMAL_COND_PROB / PLANTAR_COND_PROB dict in Section 1."
        )
    print(f"{name}: config labels {sorted(configured_labels)} all present in data {sorted(actual_labels)}")

check_labels_match("Thermal", thermal_df, THERMAL_COND_PROB)
check_labels_match("Plantar", plantar_df, PLANTAR_COND_PROB)

Thermal: config labels ['Control', 'DM'] all present in data ['Control', 'DM']
Plantar: config labels ['flat foot', 'normal', 'over-arch'] all present in data ['flat foot', 'normal', 'over-arch']


In [13]:
paired_frames = []
for split_name in ["train", "holdout"]:
    rgb_split = rgb_df[rgb_df["split"] == split_name].reset_index(drop=True)
    thermal_split = thermal_df[thermal_df["split"] == split_name].reset_index(drop=True)
    plantar_split = plantar_df[plantar_df["split"] == split_name].reset_index(drop=True)

    print(f"\nGenerating pseudo-pairs for '{split_name}' split...")
    print(f"  RGB: {len(rgb_split)} | Thermal: {len(thermal_split)} | Plantar: {len(plantar_split)}")

    paired = generate_pseudo_paired_split(rgb_split, thermal_split, plantar_split, split_name, rng)
    paired_frames.append(paired)

paired_df = pd.concat(paired_frames, ignore_index=True)
print(f"\nTotal pseudo-paired cases: {len(paired_df)}")


Generating pseudo-pairs for 'train' split...
  RGB: 1196 | Thermal: 8520 | Plantar: 4621

Generating pseudo-pairs for 'holdout' split...
  RGB: 299 | Thermal: 422 | Plantar: 341

Total pseudo-paired cases: 1495


## 5. Sanity checks

Before this feeds into fusion, confirm: (a) fused-label distribution matches the original RGB distribution exactly (pairing must not distort the target class balance), (b) the achieved conditional co-occurrence roughly matches the configured targets, (c) no split leakage.

In [14]:
# (a) fused label distribution should exactly match RGB source distribution per split
for split_name in ["train", "holdout"]:
    orig = rgb_df[rgb_df["split"] == split_name]["rgb_label"].value_counts(normalize=True).sort_index()
    fused = paired_df[paired_df["split"] == split_name]["fused_label"].value_counts(normalize=True).sort_index()
    print(f"[{split_name}] RGB label dist:   {orig.to_dict()}")
    print(f"[{split_name}] fused label dist: {fused.to_dict()}")
    assert (orig.sort_index().values == fused.sort_index().values).all(), "Fused label distribution drifted from RGB source!"

print("\nFused label distribution matches RGB source exactly. \n")

# (b) achieved vs target conditional co-occurrence
print("Achieved Thermal | RGB-label co-occurrence (train split):")
print(pd.crosstab(paired_df[paired_df.split == "train"]["rgb_label"],
                   paired_df[paired_df.split == "train"]["thermal_label"], normalize="index").round(2))

print("\nAchieved Plantar | RGB-label co-occurrence (train split):")
print(pd.crosstab(paired_df[paired_df.split == "train"]["rgb_label"],
                   paired_df[paired_df.split == "train"]["plantar_label"], normalize="index").round(2))

# (c) with-replacement usage rate (high = one modality's class pool is much smaller than RGB's; worth knowing)
print(f"\nThermal sampled with replacement: {paired_df['thermal_used_with_replacement'].mean()*100:.1f}% of cases")
print(f"Plantar sampled with replacement: {paired_df['plantar_used_with_replacement'].mean()*100:.1f}% of cases")

[train] RGB label dist:   {0: 0.1806020066889632, 1: 0.8193979933110368}
[train] fused label dist: {0: 0.1806020066889632, 1: 0.8193979933110368}
[holdout] RGB label dist:   {0: 0.1806020066889632, 1: 0.8193979933110368}
[holdout] fused label dist: {0: 0.1806020066889632, 1: 0.8193979933110368}

Fused label distribution matches RGB source exactly. 

Achieved Thermal | RGB-label co-occurrence (train split):
thermal_label  Control    DM
rgb_label                   
0                 0.67  0.33
1                 0.14  0.86

Achieved Plantar | RGB-label co-occurrence (train split):
plantar_label  flat foot  normal  over-arch
rgb_label                                  
0                   0.14    0.72       0.14
1                   0.40    0.19       0.40

Thermal sampled with replacement: 0.0% of cases
Plantar sampled with replacement: 0.1% of cases


## 6. Save outputs

These CSVs are the direct input to the Phase 4 late-fusion notebook — join `thermal_row_idx`/`plantar_row_idx` back against `thermal_features.csv` / `plantar_features.csv` to pull feature vectors, and use `rgb_path` for EfficientNet-B3 inference.

In [15]:
train_out = paired_df[paired_df["split"] == "train"].reset_index(drop=True)
holdout_out = paired_df[paired_df["split"] == "holdout"].reset_index(drop=True)

train_out.to_csv("/content/paired_train.csv", index=False)
holdout_out.to_csv("/content/paired_holdout.csv", index=False)

print(f"Saved /content/paired_train.csv   ({len(train_out)} rows)")
print(f"Saved /content/paired_holdout.csv ({len(holdout_out)} rows)")
print("\nCross-Modal Validation Strategy (Path A) — pairing complete. Ready for Phase 4 (Late Fusion Implementation).")

Saved /content/paired_train.csv   (1196 rows)
Saved /content/paired_holdout.csv (299 rows)

Cross-Modal Validation Strategy (Path A) — pairing complete. Ready for Phase 4 (Late Fusion Implementation).
